# 05 — Ablation Calismasi (Adim 7)

K1 (karbon), K2 (kur), K3 (geo) feature gruplarinin modelin karar mekanizmasinda load-bearing oldugunu kanitlamak.

**Senaryolar (7):** full + without_fx, without_geo, without_carbon, without_demand, without_quality, without_cost

**Cikti:** `data/processed/ablation_results.json`

In [ ]:
from __future__ import annotations
import sys, json, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

In [ ]:
from ml.src.ablation import run_ablation

df = pd.read_parquet(REPO_ROOT / 'data/processed/training_set_v1.parquet')

# gbm_results.json'dan best_kind oku
gbm = json.loads((REPO_ROOT / 'data/processed/gbm_results.json').read_text())
best_p = gbm['profit']['best_model']
best_r = gbm['route']['best_model']
print('best profit kind:', best_p)
print('best route kind:', best_r)

In [ ]:
t0 = time.time()
results = run_ablation(df, best_profit_kind=best_p, best_route_kind=best_r, seed=42)
print(f'Total time: {time.time()-t0:.1f}s')

out_path = REPO_ROOT / 'data/processed/ablation_results.json'
out_path.write_text(json.dumps(results, indent=2, default=float))
print('saved:', out_path)

In [ ]:
# Tablo: scenario, profit_rmse, delta_pct, route_macro_f1
rows = []
for name, m in results.items():
    rows.append({
        'scenario': name,
        'profit_rmse': m['profit_rmse'],
        'rmse_delta_pct': m['rmse_delta_pct'],
        'route_macro_f1': m['route_macro_f1'],
    })
df_res = pd.DataFrame(rows).sort_values('rmse_delta_pct', ascending=False, key=abs)
df_res

In [ ]:
# Bar chart: feature group -> RMSE delta
abl = df_res[df_res['scenario'] != 'full_features'].sort_values('rmse_delta_pct', ascending=True)
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if v > 10 else '#ff7f0e' if v > 0 else '#2ca02c' for v in abl['rmse_delta_pct']]
ax.barh(abl['scenario'], abl['rmse_delta_pct'], color=colors)
ax.axvline(10, ls='--', color='gray', lw=1, label='10% threshold')
ax.set_xlabel('RMSE delta (%) — full_features = 0')
ax.set_title('Ablation: feature group importance')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Yorum cumleleri
for name, m in results.items():
    print(f'\n--- {name} ---')
    print(f"  profit_rmse={m['profit_rmse']:,.0f}  delta={m['rmse_delta_pct']:+.1f}%  route_macro_f1={m['route_macro_f1']:.3f}")
    print(f"  {m['interpretation']}")